# Which Data Helps? Block-by-Block Test of the Master Table

The master table (`features_daily_v2.parquet`) has the original **25 features** plus **46 new ones** from 9 new sources. This notebook asks: **which sources actually help predict next-day stock returns?**

**One simple model, many input sets.** The model is the same every time: a **linear model** (ridge regression) that ranks all ~500 S&P 500 stocks each day. Only the inputs change:

| Test | Inputs | Question it answers |
|---|---|---|
| **Base** | the original 25 features | the starting point |
| **Base + one block** (×9) | base + options, + liquidity, + short selling, ... | does this source add anything **on top of** the base? |
| **All** | base + all 9 blocks (71 features) | the whole thing |
| **All − one block** (×9) | everything except one source | does this source add anything that the **other sources don't** already cover? |
| **Options lagged 1 day** | options block using the previous day's options data | is the options gain real, or a closing-price timing effect? (Section 3b) |
| **Deployable** | only sources available live at low cost | could a live version keep most of the gain? |
| **All, nonlinear** | all 71 features in a gradient-boosted tree model | do interactions between features add more? |

**Setup (same rules as before):**
- **Universe:** S&P 500 members on each decision day (point-in-time).
- **Inputs:** every feature is **ranked across stocks each day** and scaled to −0.5 … +0.5 (0 = median); missing = 0; 0/1 flags stay 0/1 (shifted to ±0.5). Ranking needs only that day's cross-section, so there is no look-ahead, and it makes features with very different units comparable.
- **Target:** next-day excess return, standardized across stocks each day.
- **Walk-forward:** re-fit **every year** on the previous 5 years (every other day, to save time); predict every day of the next year. 2006–2022.
- **Judging:** decide on **2006–2020** only (rank IC, Newey–West t, gross Sharpe of the daily top-20% / bottom-20% long–short). 2021–2022 is shown separately at the end as the check.

**A warning about many tests:** ~30 versions are compared here, so one or two will look good by luck. Trust effects with **t above about 3** that also hold up in 2021–2022.

Takes about 4 minutes.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

import pit_common as pc
from gdbn_utils import RESULTS

OUT_DIR = RESULTS
FIT_YEARS, YEARS = 5, range(2006, 2023)
RUN_NONLINEAR = True        # gradient-boosted trees for "All" and "Base" (about 10 extra minutes)

BLOCKS = {
    "base": ["exret_1d", "exret_5d", "exret_20d", "mom_12_1", "overnight", "intraday",
             "vol_1d", "range", "rvol_20", "beta_60", "ivol_60", "volume", "turnover_20", "amihud_20", "size",
             "sentiment", "has_news", "attention", "surprise",
             "days_since_earn", "sue", "bm", "gp", "lev", "ag"],
    "options": ["iv_atm", "iv_spread", "iv_skew", "iv_chg_5d", "iv_rv", "has_options", "pc_volume", "os_volume"],
    "liquidity": ["spread", "spread_20"],
    "short selling": ["lend_util", "lend_cost", "lend_fee", "lend_util_chg", "has_lending",
                      "si_ratio", "si_days_to_cover", "si_change", "has_short"],
    "analysts": ["an_coverage", "an_revision", "an_dispersion", "an_updown", "has_analysts", "rec_mean", "rec_change", "ibes_sue"],
    "institutions": ["io_ratio", "io_n", "io_change", "has_inst"],
    "insiders": ["ins_buys_90", "ins_sells_90", "ins_net_90", "has_insiders"],
    "events": ["ev_count", "ev_guidance", "ev_buyback", "ev_mna", "ev_mgmt", "ev_lawsuit"],
    "credit": ["cds_5y", "cds_chg_5d", "has_cds"],
    "text": ["txt_similarity", "has_text"],
}
NEW_BLOCKS = [b for b in BLOCKS if b != "base"]
ALL = [f for b in BLOCKS.values() for f in b]
# Deployable at low cost (see the vault note): prices / volume, SEC filings (fundamentals, insiders, 13F, 10-K text,
# 8-K events as a stand-in for Capital IQ), options and borrow rates from a broker, FINRA short interest.
# Excluded: RavenPack news (base sentiment / attention), I/B/E/S analysts, Markit CDS.
NEWS = ["sentiment", "has_news", "attention", "surprise"]
DEPLOYABLE = [f for f in BLOCKS["base"] if f not in NEWS] + [f for b in ["options", "liquidity", "short selling", "institutions",
                                                                         "insiders", "events", "text"] for f in BLOCKS[b]]
BINARY = [f for f in ALL if f.startswith("has_") or (f.startswith("ev_") and f != "ev_count")]
print(f"{len(ALL)} features in {len(BLOCKS)} blocks; deployable set: {len(DEPLOYABLE)} features")

D = pc.load()
t0 = time.time()
df = pd.read_parquet(pc.BUILT / "features_daily_v2.parquet", columns=["date", "permno", "in_sp500", "target_next"] + ALL)
df = df[df["date"] <= pc.EVAL_END].sort_values(["permno", "date"]).reset_index(drop=True)
g = df.groupby("permno")
LAGGED = [f"{f}_lag" for f in BLOCKS["options"]]
for f in BLOCKS["options"]:
    df[f"{f}_lag"] = g[f].shift(1)                       # the previous trading day's options data
df["next_overnight"] = g["overnight"].shift(-1)          # for the timing check (Section 3b)
df["next_intraday"] = g["intraday"].shift(-1)
df = df[df["in_sp500"] > 0].reset_index(drop=True)
print(f"S&P 500 member stock-days: {len(df):,} ({df['date'].min().date()} to {df['date'].max().date()}; {time.time() - t0:.0f}s)")

## 1. Rank the features across stocks each day

In [ ]:
t0 = time.time()
USE = ALL + LAGGED
BIN = BINARY + [f for f in LAGGED if f.startswith("has_")]
cont = [f for f in USE if f not in BIN]
Xr = df.groupby("date")[cont].rank(pct=True).sub(0.5).fillna(0.0).astype("float32")      # -0.5 ... +0.5, missing = median
Xb = df[BIN].fillna(0.0).sub(0.5).astype("float32")                                         # flags: -0.5 / +0.5
X = pd.concat([Xr, Xb], axis=1)[USE]
COL = {f: i for i, f in enumerate(USE)}
X = X.values
g = df.groupby("date")["target_next"]
Y = ((df["target_next"] - g.transform("mean")) / g.transform("std")).clip(-5, 5).values.astype("float32")
DAY = df["date"].values
PERM = df["permno"].values
udays = pd.DatetimeIndex(np.unique(DAY))
day_pos = udays.get_indexer(DAY)
print(f"Feature matrix: {X.shape[0]:,} x {X.shape[1]} ({X.nbytes / 1e6:.0f} MB, {time.time() - t0:.0f}s)")

# Training rows (every other day of the past 5 years, target known) and prediction rows, for each year
SPLITS = {}
for Y_ in YEARS:
    a, b = pd.Timestamp(f"{Y_ - FIT_YEARS}-01-01"), pd.Timestamp(f"{Y_}-01-01")
    win_days = udays[(udays >= a) & (udays < b)][:-1]                    # last day's target falls in year Y_
    keep_days = win_days[::2].values
    train = np.where(np.isin(DAY, keep_days) & np.isfinite(Y))[0]
    test = np.where(pd.DatetimeIndex(DAY).year == Y_)[0]
    SPLITS[Y_] = (train, test)
print(f"Training rows per year: {np.mean([len(t) for t, _ in SPLITS.values()]):,.0f}")

## 2. Walk-forward for every input set

Each prediction made at day $t$'s close is stored under day $t+1$ (the day it predicts), the same convention as the other notebooks.

In [ ]:
NEXT = pd.Series(D.days[np.minimum(D.days.searchsorted(udays, side="right"), len(D.days) - 1)], index=udays)


def walk_forward(features, model="ridge"):
    cols = [COL[f] for f in features]
    pred = np.full(len(Y), np.nan, dtype="float32")
    coefs = []
    for Y_, (train, test) in SPLITS.items():
        if model == "ridge":
            m = Ridge(alpha=1.0).fit(X[np.ix_(train, cols)], Y[train])
            coefs.append(m.coef_)
        else:
            m = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=200,
                                              l2_regularization=1.0, random_state=Y_).fit(X[np.ix_(train, cols)], Y[train])
        pred[test] = m.predict(X[np.ix_(test, cols)])
    s = pd.DataFrame({"date": NEXT.values[day_pos], "permno": PERM, "score": pred}).dropna()
    s = s[s["date"] > pd.Timestamp(f"{min(YEARS)}-01-01")]
    wide = s.pivot_table(index="date", columns="permno", values="score").reindex(columns=D.RET.columns)
    return wide, (np.mean(coefs, axis=0) if coefs else None)


SPECS = {"Base": BLOCKS["base"]}
for b in NEW_BLOCKS:
    SPECS[f"Base + {b}"] = BLOCKS["base"] + BLOCKS[b]
SPECS["All"] = ALL
for b in NEW_BLOCKS:
    SPECS[f"All − {b}"] = [f for f in ALL if f not in BLOCKS[b]]
SPECS["Deployable"] = DEPLOYABLE
SPECS["Base + options (lagged 1 day)"] = BLOCKS["base"] + LAGGED
SPECS["All (options lagged 1 day)"] = [f for f in ALL if f not in BLOCKS["options"]] + LAGGED
SPECS["Deployable (options lagged 1 day)"] = [f for f in DEPLOYABLE if f not in BLOCKS["options"]] + LAGGED

scores, coef_all = {}, None
t0 = time.time()
for name, feats in SPECS.items():
    scores[name], c = walk_forward(feats)
    if name == "All":
        coef_all = pd.Series(c, index=feats)
    print(f"  {name:28s} {len(feats):3d} features ({time.time() - t0:.0f}s)")
if RUN_NONLINEAR:
    for name, feats in [("Base, nonlinear", BLOCKS["base"]), ("All, nonlinear", ALL)]:
        scores[name], _ = walk_forward(feats, model="gbm")
        print(f"  {name:28s} {len(feats):3d} features ({time.time() - t0:.0f}s)")

## 3. Results on 2006–2020 (the period used to decide)

**IC** = average daily rank correlation between predictions and next-day excess returns; **IC t** = Newey–West t-statistic; **gross Sharpe** = annualized Sharpe of buying the top 20% and shorting the bottom 20% every day, before costs; **Δ IC vs Base** / **vs All** = the change from adding / removing the block.

In [ ]:
def evaluate(score, a, b):
    s = score.loc[a:b]
    ic = pc.rank_ic(s, pc.forward_excess(D, s, 1))
    sm = pc.summary(D, pc.daily_portfolio(s), a, b)
    return pd.Series({"IC": ic.mean(), "IC t": pc.nw_t(ic), "Sharpe gross": sm["Sharpe gross"],
                      "turnover/day": sm["turnover/day"], "breakeven bps": sm["breakeven bps"]})


DEV, VAL = pc.PERIODS["2006–2020 walk-forward"], pc.PERIODS["2021–2022 validation"]
dev = pd.DataFrame({n: evaluate(s, *DEV) for n, s in scores.items()}).T
dev.insert(0, "features", [len(SPECS[n]) if n in SPECS else (len(ALL) if "All" in n else len(BLOCKS["base"])) for n in dev.index])
base_ic, all_ic = dev.loc["Base", "IC"], dev.loc["All", "IC"]
dev["Δ IC vs Base"] = np.where(dev.index.str.startswith("Base +"), dev["IC"] - base_ic, np.nan)
dev["Δ IC vs All"] = np.where(dev.index.str.startswith("All −"), dev["IC"] - all_ic, np.nan)
print("===== 2006–2020 =====")
print(dev.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
add = dev[dev.index.str.startswith("Base +")]["Δ IC vs Base"].sort_values()
axes[0].barh([n.replace("Base + ", "") for n in add.index], add.values * 1e3, color=np.where(add > 0, "tab:green", "tab:red"))
axes[0].set_title("Adding one block to Base: change in IC (×1000)"); axes[0].axvline(0, color="black", lw=0.8)
rem = dev[dev.index.str.startswith("All −")]["Δ IC vs All"].sort_values()
axes[1].barh([n.replace("All − ", "") for n in rem.index], rem.values * 1e3, color=np.where(rem < 0, "tab:green", "tab:red"))
axes[1].set_title("Removing one block from All: change in IC (×1000)\n(more negative = the block matters more)")
axes[1].axvline(0, color="black", lw=0.8)
for name in ["Base", "All", "Deployable"] + (["All, nonlinear"] if RUN_NONLINEAR else []):
    s = scores[name].loc[:pc.EVAL_END]
    axes[2].plot(pc.rank_ic(s, pc.forward_excess(D, s, 1)).cumsum(), label=name)
axes[2].axvline(pd.Timestamp("2021-01-01"), color="gray", lw=1)
axes[2].set_title("Cumulative rank IC (line = validation starts)"); axes[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3b. Is the options gain real? A timing check

Options-implied volatilities are computed against the stock's **closing price**. If that closing price is slightly off (for example, the last trade happened at the bid), calls and puts look mispriced against it, and the stock "bounces back" overnight. A feature built from closing option quotes can then predict the next **overnight** move without containing any real information, and nobody could trade on it at that close (the options data only exists after the market has closed).

Two tests:
- **Where does the predictability show up?** Overnight (close → next open) vs during the next day (open → close).
- **Does it survive with one day older options data?** A real information signal should fade slowly; a closing-price artifact should vanish.

In [ ]:
def ic_of(x, y, data):
    s = data[["date", x, y]].dropna()
    r = s.groupby("date")[[x, y]].rank()
    r = r - r.groupby(s["date"]).transform("mean")
    G = lambda v: v.groupby(s["date"]).sum()
    v = G(r[x] * r[y]) / np.sqrt(G(r[x] ** 2) * G(r[y] ** 2))
    return v.replace([np.inf, -np.inf], np.nan).mean()


sub = df[df["date"].between("2006-01-01", pc.TUNE_END)]
rows = {}
for f in ["iv_spread", "iv_skew", "iv_chg_5d", "iv_rv", "pc_volume", "os_volume", "exret_1d"]:
    rows[f] = {"IC next day": ic_of(f, "target_next", sub), "IC next overnight": ic_of(f, "next_overnight", sub),
               "IC next daytime": ic_of(f, "next_intraday", sub),
               "IC next day, data 1 day older": ic_of(f"{f}_lag", "target_next", sub) if f"{f}_lag" in sub else np.nan}
print(pd.DataFrame(rows).T.round(4).to_string())

**Reading it:** if a feature's IC is mostly **overnight** and drops to about zero with one-day-older data, its "skill" is largely the closing-price bounce, not information. The **lagged** model rows in the table above (and in Section 5) are then the honest estimate of what the options block adds. (`exret_1d`, yesterday's return reversal, is shown for comparison: part of it is the same bounce effect.)

**Reading it:**
- **Left (adding):** how much each source improves the base model on its own.
- **Middle (removing):** how much each source adds that **no other source** already covers. A block can look good when added alone but add little to "All" if another block carries the same information (e.g., options and short selling both measure bearish demand).
- **Right:** does the whole thing beat the base model consistently over time, or only in a few episodes?

## 4. Which individual features does the full model lean on?

Average weight of each feature in the "All" ridge model (features are ranked, so weights are comparable). Positive = higher values predict higher next-day excess returns.

In [ ]:
top = pd.concat([coef_all.sort_values().head(12), coef_all.sort_values().tail(12)])
block_of = {f: b for b, fs in BLOCKS.items() for f in fs}
fig, ax = plt.subplots(figsize=(9, 7))
colors = [plt.cm.tab10(list(BLOCKS).index(block_of[f]) % 10) for f in top.index]
ax.barh([f"{f}  ({block_of[f]})" for f in top.index], top.values, color=colors)
ax.axvline(0, color="black", lw=0.8); ax.set_title("Largest weights in the 'All' model (average over the yearly fits)")
plt.tight_layout(); plt.show()

## 5. The check: 2021–2022 (look only after deciding on 2006–2020)

In [ ]:
val = pd.DataFrame({n: evaluate(s, *VAL) for n, s in scores.items()}).T
both = pd.concat({"2006–2020": dev[["IC", "IC t", "Sharpe gross"]], "2021–2022": val[["IC", "IC t", "Sharpe gross"]]}, axis=1)
print(both.round(4).to_string())

**Reading it:** a block is worth keeping if it helped in 2006–2020 **and** the model that includes it is still ahead in 2021–2022. Two years are noisy (about 500 days), so look for the same direction rather than the same size.

In [ ]:
pd.concat({"2006–2020": dev, "2021–2022": val}, names=["period", "model"]).to_csv(OUT_DIR / "feature_blocks.csv")
print("Saved to", OUT_DIR / "feature_blocks.csv")